<a href="https://colab.research.google.com/github/casper-justus/swahili-gpt/blob/main/inference_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇰🇪 Swahili GPT — Interactive Inference Demo

Generate Kiswahili text from a trained checkpoint, entirely in Colab.
No local setup required — just mount your Google Drive and run all cells.

---

## 📋 Before You Start

### ✅ Step 1: Enable the GPU
> Go to **Runtime → Change runtime type → T4 GPU → Save**.
> The model will still load on CPU but generation will be very slow.

### ✅ Step 2: Make sure your checkpoint exists on Drive
> Your checkpoint folder should be at:
> `/MyDrive/MiniGPT_Kiswahili_Checkpoints/`
> with a subfolder like `5000/` or `10000/` inside it.
> Also ensure `kenya_tokenizer.json` is in that same folder.

### ✅ Step 3: Run all cells top to bottom
> Then scroll to **Cell 5** to type your prompt and generate text interactively.


## Cell 1 — Mount Google Drive
Connects this notebook to your Google Drive so the checkpoint and tokenizer can be loaded.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints"

# Verify the checkpoint folder exists
if not os.path.exists(CKPT_DIR):
    raise FileNotFoundError(f"Checkpoint folder not found: {CKPT_DIR}\nMake sure you have trained the model first.")

print(f"✅ Drive mounted. Checkpoint dir: {CKPT_DIR}")
print(f"   Contents: {os.listdir(CKPT_DIR)}")


Mounted at /content/drive
✅ Drive mounted. Checkpoint dir: /content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints
   Contents: ['kenyan_tokenizer.json', '45000', '50000', '55000']


## Cell 2 — Install Dependencies
Installs the same pinned library versions used during training to guarantee compatibility.
> ⚠️ This takes 2–3 minutes. The long output is normal.


In [ ]:
!pip uninstall -y jax jaxlib flax optax orbax-checkpoint -q
!pip install "jax[cuda12]" flax optax orbax-checkpoint tokenizers -q
print("✅ Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.0/403.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6

## Cell 3 — Load Model Architecture & Tokenizer
Rebuilds the exact same model architecture that was used during training, then loads the saved weights from your Drive checkpoint.

> ⚠️ The model config below **must match your training hyperparameters exactly**.
> If you changed `EMB_SIZE`, `NUM_HEADS`, or `NUM_LAYERS` during training, update them here too.


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tokenizers import Tokenizer

# ── Model config (must match training) ─────────────────────────────
SEQ_LEN    = 1024
EMB_SIZE   = 512
NUM_HEADS  = 8
NUM_LAYERS = 6

# ── Architecture definition ──────────────────────────────────
class Block(nnx.Module):
    def __init__(self, emb_size, num_heads, rngs):
        self.ln_1 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(num_heads=num_heads, in_features=emb_size, decode=False, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.mlp  = nnx.Sequential(
            nnx.Linear(emb_size, 4 * emb_size, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * emb_size, emb_size, rngs=rngs)
        )
    def __call__(self, x, mask):
        x = x + self.attn(self.ln_1(x), mask=mask)
        x = x + self.mlp(self.ln_2(x))
        return x

class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, seq_len, emb_size, num_heads, num_layers, rngs):
        self.token_emb = nnx.Embed(vocab_size, emb_size, rngs=rngs)
        self.pos_emb   = nnx.Embed(seq_len, emb_size, rngs=rngs)
        self.blocks    = nnx.Sequential(*[Block(emb_size, num_heads, rngs) for _ in range(num_layers)])
        self.ln_f      = nnx.LayerNorm(emb_size, rngs=rngs)
        self.lm_head   = nnx.Linear(emb_size, vocab_size, rngs=rngs)
    def __call__(self, idx):
        b, t = idx.shape
        pos  = jnp.arange(0, t, dtype=jnp.int32)[None, :]
        x    = self.token_emb(idx) + self.pos_emb(pos)
        mask = nnx.make_causal_mask(jnp.ones((b, t)))
        for block in self.blocks.layers:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))

# ── Load tokenizer ──────────────────────────────────────────
print("Loading tokenizer...")
tokenizer  = Tokenizer.from_file(f"{CKPT_DIR}/kenyan_tokenizer.json")
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"✅ Tokenizer loaded. Vocab size: {VOCAB_SIZE}")

# ── Build model shell ──────────────────────────────────────────
print("Building model...")
rngs  = nnx.Rngs(0)
model = MiniGPT(VOCAB_SIZE, SEQ_LEN, EMB_SIZE, NUM_HEADS, NUM_LAYERS, rngs)

# ── Restore weights from checkpoint ───────────────────────────
mngr = ocp.CheckpointManager(CKPT_DIR, options=ocp.CheckpointManagerOptions(max_to_keep=3))

if mngr.latest_step() is None:
    raise FileNotFoundError("No checkpoint found in Drive. Train the model first using the training notebook.")

_, model_state = nnx.split(model)
restored       = mngr.restore(mngr.latest_step(), args=ocp.args.StandardRestore({'model': model_state}))
nnx.update(model, restored['model'])
print(f"✅ Weights loaded from step {mngr.latest_step()}.")
print(f"   Detected device: {jax.devices()[0]}")


## Cell 4 — Generation Function
Defines the `generate()` function using **top-k sampling with temperature scaling**.

**How the sampling parameters work:**
| Parameter | Effect | Recommended range |
|---|---|---|
| `temperature` | Controls randomness. Lower = repetitive but safe. Higher = creative but risky. | `0.7 – 1.0` |
| `top_k` | Only picks from the top K most likely next tokens. | `30 – 60` |
| `max_new_tokens` | How many new tokens to generate after the prompt. | `50 – 200` |

> ⚠️ **First generation call will take ~30–60 seconds** for JAX JIT compilation. All subsequent calls will be instant.


In [ ]:
import jax
import jax.numpy as jnp

def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    """Generate Kiswahili text from a prompt using top-k sampling."""
    rng    = jax.random.PRNGKey(42)
    tokens = tokenizer.encode(prompt).ids

    for _ in range(max_new_tokens):
        ctx    = tokens[-SEQ_LEN:]
        logits = model(jnp.array([ctx]))[0, -1, :] / temperature
        top_logits, top_idx = jax.lax.top_k(logits, top_k)
        chosen  = jax.random.categorical(rng, top_logits)
        next_tok = int(top_idx[chosen])
        tokens.append(next_tok)
        rng = jax.random.fold_in(rng, next_tok)

    return tokenizer.decode(tokens)

print("✅ generate() function ready.")
print("   First call will take ~30-60s for JIT compilation. Subsequent calls are instant.")


## Cell 5 — ✨ Generate Text (Edit This Cell)
This is the cell you run repeatedly. Edit `PROMPT`, `TEMPERATURE`, and `MAX_TOKENS` to experiment.

**Prompt ideas to try:**
- `"Habari za asubuhi"` (Good morning news)
- `"Rais wa Kenya alisema"` (The President of Kenya said)
- `"Watoto wa shule"` (School children)
- `"Mvua ilikuwa inanyesha"` (It was raining)
- `"Siku moja, mtu mmoja"` (One day, a person)


In [ ]:
# ✏️ Edit these values, then run this cell
PROMPT         = "Habari za asubuhi"  # Your Kiswahili prompt
MAX_TOKENS     = 100                  # Number of new tokens to generate
TEMPERATURE    = 0.8                  # 0.1 = safe/repetitive, 1.5 = creative/wild
TOP_K          = 40                   # Sample from top 40 most likely tokens

print(f"Prompt : \"{PROMPT}\"")
print(f"Tokens : {MAX_TOKENS} | Temp: {TEMPERATURE} | Top-k: {TOP_K}")
print("-" * 60)

output = generate(model, tokenizer, PROMPT, MAX_TOKENS, TEMPERATURE, TOP_K)

print(output)
print("-" * 60)


## Cell 6 — Batch Test Multiple Prompts
Run several prompts at once to quickly evaluate model quality across different topics.


In [ ]:
test_prompts = [
    "Habari za asubuhi",
    "Rais wa Kenya alisema",
    "Watoto wa shule",
    "Mvua ilikuwa inanyesha",
    "Siku moja, mtu mmoja",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print('='*60)
    output = generate(model, tokenizer, prompt, max_new_tokens=60, temperature=0.8, top_k=40)
    print(output)
